In [1]:
import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
import os
import json
import sglang as sgl
from llavaguard.taxonomy.PEGI.PEGI_Graph import remove_numbers_from_categories, get_policy_assessment, policy_graph, policy_graph_to_safety_policy_v2, get_rating
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_content_categories, get_content_categories_with_examples, get_content_categories_with_numbers, get_policy_intro, get_safety_categories, policy_graph, policy_graph_to_safety_policy
local_data_dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo'

In [ ]:
#extract category assessments from majority vote

data = f'/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/majority_vote.csv'
output_dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'
df = pd.read_csv(data)
df = df[df['voting_mechanism'] == 'majority_vote']
categories = df.columns[3:]
extracted_data = []
for index, row in df.iterrows():
    #print('index = ', index)
    for category in categories:
        if row[category] == 1:
            extracted_data.append({
                'image_name': row['sample_id'],
                'file_path': row['im_path'],
                'category': category,
            })
result_df = pd.DataFrame(extracted_data)
result_df.to_csv(f'{output_dir}/extracted_categories_numbers.csv', index=False)

index =  0
index =  3
index =  6
index =  9
index =  12
index =  15
index =  18
index =  21
index =  24
index =  27
index =  30
index =  33
index =  36
index =  39
index =  42
index =  45
index =  48
index =  51
index =  54
index =  57
index =  60
index =  63
index =  66
index =  69
index =  72
index =  75
index =  78
index =  81
index =  84
index =  87
index =  90
index =  93
index =  96
index =  99
index =  102
index =  105
index =  108
index =  111
index =  114
index =  117
index =  120
index =  123
index =  126
index =  129
index =  132
index =  135
index =  138
index =  141
index =  144
index =  147
index =  150
index =  153
index =  156
index =  159
index =  162
index =  165
index =  168
index =  171
index =  174
index =  177
index =  180
index =  183
index =  186
index =  189
index =  192
index =  195
index =  198
index =  201
index =  204
index =  207
index =  210
index =  213
index =  216
index =  219
index =  222
index =  225
index =  228
index =  231
index =  234
index =  23

In [ ]:
#change image paths to correct ones

import pandas as pd

dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'

df = pd.read_csv(f'{dir}/extracted_categories.csv', header=None, names=["image_id", "path", "category"])


old_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/lhelff/ds/LlavaGuard/data/"
new_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/"


df["path"] = df["path"].str.replace(old_prefix, new_prefix, regex=False)

df.to_csv(f'{dir}/extracted_categories.csv', index=False, header=False)



In [3]:
#returns policy, designed for the specific PEGI-Rating of this category

get_rating("Financial Crimes")
    

18

In [1]:
from transformers import AutoProcessor, Llama4ForConditionalGeneration, AutoTokenizer
import torch
from PIL import Image


model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/envs/llama-facory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards:  18%|█▊        | 9/50 [00:30<02:20,  3.43s/it]


KeyboardInterrupt: 

In [ ]:
#testing llama4 with transformers, only text input
import time

#t0 = time.perf_counter()
import transformers
#print("Import transformers took", time.perf_counter() - t0, "seconds")

#t0 = time.perf_counter()
from transformers import AutoTokenizer
#print("Import AutoTokenizer took", time.perf_counter() - t0, "seconds")

#t0 = time.perf_counter()
from transformers import AutoProcessor

#t0 = time.perf_counter()
from transformers import Llama4ForConditionalGeneration
#print("Import Llama4ForConditionalGeneration took", time.perf_counter() - t0, "seconds")

####

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", 
    torch_dtype=torch.bfloat16,
)

outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[-1] :])
print(outputs[0])

In [ ]:
#generating rationales
import os
import argparse
import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import torch
print("before transformers")
from transformers import AutoProcessor, Llama4ForConditionalGeneration
print("after transformers")
from PIL import Image
from llavaguard_config import local_image_dirs, local_data_dir

import pandas as pd
import json

#loading model 

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

#generating rationales for all the images 


processor = AutoProcessor.from_pretrained(model_id)
local_data_dir = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo" 
csv_path      = f"{local_data_dir}/data/annotations/auto_generated_annotations/v0_2/extracted_categories_stripped.csv"
output_path   = f"{local_data_dir}/data/PEGI-LlavaGuard-DS/v2/all_data.json"
model_id      = "meta-llama/Llama-4-Scout-17B-16E-Instruct"


df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from:\n  {csv_path}")
#df = df.tail(5)
df = df.head(5)
total = len(df)
all_outputs = []
checkpoint_every = 2

for idx, row in df.iterrows():
    sample_id = row["image_name"]
    print(row["file_path"])
    img_path  = row["file_path"]
    category  = row["category"]

    prompt = (
        f"Describe the image and how it relates to the following PEGI rating safety-category - {category} - in 2 sentences."
    )
    print(f"\n[Sample {idx + 1} / {len(df)}]")
    #print(img_path)
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(**inputs, max_new_tokens=200)

    start = inputs["input_ids"].shape[-1]
    rationale_text = processor.batch_decode(outputs[:, start:])[0].strip()
    #print("  →  Generated rationale:")
    #print(f"    {rationale_text}")

    all_outputs.append({
        "image_name": sample_id,
        "image_path": img_path,
        "category":   category,
        "rationale":  rationale_text
    })
    if (idx + 1) % checkpoint_every == 0 or idx + 1 == total:
        parent_dir = os.path.dirname(output_path)
        os.makedirs(parent_dir, exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(all_outputs, f, indent=4)
        print(f"  → [Checkpoint] Wrote {idx+1} / {total} entries to {output_path}", flush=True)

#parent_dir = os.path.dirname(output_path)
#os.makedirs(parent_dir, exist_ok=True)
#
#with open(output_path, "w") as f:
#    json.dump(all_outputs, f, indent=4)

print(f"\n Written {len(all_outputs)} entries to:\n {output_path}")

    

In [45]:
from llavaguard_config import local_image_dirs, local_data_dir
print(local_data_dir)

/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo


In [3]:
text = policy_graph_to_safety_policy(18)
print(text)

Provide a safety assessment for the user content. Determine compliance or violation against the safety policy by reviewing the following categories:
S1: Violence
Permitted content categories:
- "1. Educational Content" e.g. Charts on violence statistics, and trends, Informational content on prevalence, causes, and effects of violence, Legal education, Memorials and statues of historical figures linked to violence
- "2. Unrealistic and Implied Violence" e.g. Non-realistic violence, Non-detailed violence
- "3. Military and Warfare" e.g. Military, Warfare, Soldiers, Combat, Battle
- "4. Mild Violence" e.g. Violence without any indication of suffering, Violence towards fantasy characters, Non-realistic violence towards human-like characters, Moderate violence towards human characters without blood
- "5. Violent Acts (animals)" e.g. Violence towards animals, Animal kingdom violence, Suffering or dead animals, Caged or suffering animals
- "6. Realistic Violence" e.g. Violence against human c

In [5]:
def assessment(compliant_rating, curr_rating):
    if (curr_rating < compliant_rating):
        return "Compliant"
    else:
        return "Review Needed"

In [ ]:
#transform dataset to LLama-factory Format
import json
import os
import csv
import json

rationale_path = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/rationales.json'
output_path   = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/all_data_v3.json'
maj_vote = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/extracted_categories_numbers.csv'

mapping = {}
with open(maj_vote, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for image_name, img_path_csv, full_cat in reader:
        number, cat_name = full_cat.split(". ", 1)
        mapping[(image_name, cat_name)] = number


with open(rationale_path, "r", encoding="utf-8") as f:
    original = json.load(f)

compliant_rating = 18
transformed = []
for item in original:
    image_id = item["image_name"]
    raw_cat  = item["category"]
    number   = mapping.get((image_id, raw_cat))

    if number:
        cat_with_number = f"{number}. {raw_cat}"
    else:
        cat_with_number = raw_cat
        print("failed to get a number")
    
    rating = get_rating(raw_cat)
    #print(rating)
    clean_rationale = item["rationale"].split("<|eot|>")[0].strip()

    gpt_value = json.dumps({
        "rationale": clean_rationale,
        "category":  cat_with_number,
        "PEGI-rating": str(rating)
    }, indent=4)

    transformed.append({
        "id":          image_id,
        "image":       item["image_path"],
        "category":    cat_with_number,
        "PEGI-rating": str(rating),
        "conversations": [
            {"from": "human", "value": policy_graph_to_safety_policy(compliant_rating)},
            {"from": "gpt",   "value": gpt_value}
        ]
    })


with open(output_path, "w", encoding="utf-8") as f:
    json.dump(transformed, f, indent=4, ensure_ascii=False)

print(f"Written {len(transformed)} entries to {output_path}")


print(json.dumps(transformed[0], indent=4, ensure_ascii=False))


18
12
12
3
16
3
16
16
16
16
16
16
18
12
12
12
16
16
16
16
12
16
12
16
16
18
12
16
16
18
16
18
12
16
16
16
16
16
7
3
12
16
16
18
18
12
18
16
12
16
16
18
16
18
12
12
12
16
12
16
16
16
7
7
16
12
16
3
16
18
16
16
18
7
16
12
16
16
16
18
12
12
16
12
16
16
16
12
16
18
16
16
12
12
16
16
16
16
18
3
12
12
16
3
3
12
12
16
16
18
16
16
3
12
7
16
7
16
18
12
16
16
12
12
12
12
12
12
12
16
12
12
16
16
-1
7
12
7
16
12
16
12
16
16
16
16
18
12
16
7
12
12
16
3
12
16
12
12
12
12
12
16
12
16
16
12
12
16
12
16
16
18
12
7
12
12
12
12
12
16
18
12
7
16
16
12
12
16
12
7
16
3
3
12
12
16
16
16
16
18
12
12
16
12
16
12
16
3
16
16
12
16
16
12
16
12
12
12
12
12
16
12
16
12
12
16
16
12
7
16
12
12
16
7
16
16
18
12
16
16
16
16
16
16
12
12
16
12
16
12
12
16
16
16
18
-1
7
16
-1
7
16
12
12
12
16
18
7
12
16
3
16
16
18
16
18
16
16
16
12
12
16
16
12
16
16
18
16
16
18
12
16
16
7
16
16
12
12
16
16
12
12
16
16
12
12
16
16
12
16
16
18
12
12
16
12
12
16
16
12
16
16
16
18
12
16
3
12
3
16
16
18
16
3
18
18
16
16
12
18
16
12
16
16
18
16

In [28]:
import json
from pprint import pprint

output_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/all_data.json"


with open(output_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records\n")

for i, rec in enumerate(data[:5]):
    print(f"Record {i} raw:")
    pprint(rec)
    print("Keys:", list(rec.keys()))
    print("-" * 40)


Loaded 2835 records

Record 0 raw:
{'PEGI-rating': '18',
 'category': 'High-Risk Financial Activities',
 'conversations': [{'from': 'human',
                    'value': 'Provide a safety assessment for the user '
                             'content. Determine compliance or violation '
                             'against the safety policy by reviewing the '
                             'following categories:\n'
                             'S1: Violence\n'
                             'Permitted content categories:\n'
                             '- "1. Educational Content" e.g. Charts on '
                             'violence statistics, and trends, Informational '
                             'content on prevalence, causes, and effects of '
                             'violence, Legal education, Memorials and statues '
                             'of historical figures linked to violence\n'
                             '- "2. Unrealistic and Implied Violence" e.g. '
         